In [1]:
# ######### used this part for fixing problems running on ARC #

import os


os.environ['HF_HOME'] = '/projects/abbott_lab/Users/ishtiaque/hfmodels/'
os.environ['HF_HUB_CACHE'] = '/projects/abbott_lab/Users/ishtiaque/hfmodels/'
os.environ['XDG_CACHE_HOME'] = '/projects/abbott_lab/Users/ishtiaque/hfmodels/'
os.environ['NB_USER'] = 'ishtiahmed'#'ishtiaqueahmedk'
os.environ['TRANSFORMERS_CACHE'] = '/projects/abbott_lab/Users/ishtiaque/hfmodels/'
os.environ['HF_DATASETS_CACHE'] = '/projects/abbott_lab/Users/ishtiaque/hfmodels/'




In [2]:
import os
import json
from PIL import Image, ImageDraw, ImageFont
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import random

import torch
import os 
import json
import random
from tqdm import tqdm
from collections import Counter 

import numpy as np
import torch


/projects/abbott_lab/Users/ishtiaque/env/hf_models_general/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/projects/abbott_lab/Users/ishtiaque/env/hf_models_general/lib/python3.10/site-packages/transformers/utils/hub.py:106: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [ ]:
import torchvision.transforms as T
# from decord import VideoReader, cpu
from PIL import Image
from torchvision.transforms.functional import InterpolationMode
from transformers import AutoModel, AutoTokenizer
import accelerate 

In [3]:

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def build_transform(input_size):
    MEAN, STD = IMAGENET_MEAN, IMAGENET_STD
    transform = T.Compose([
        T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=MEAN, std=STD)
    ])
    return transform

def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    best_ratio_diff = float('inf')
    best_ratio = (1, 1)
    area = width * height
    for ratio in target_ratios:
        target_aspect_ratio = ratio[0] / ratio[1]
        ratio_diff = abs(aspect_ratio - target_aspect_ratio)
        if ratio_diff < best_ratio_diff:
            best_ratio_diff = ratio_diff
            best_ratio = ratio
        elif ratio_diff == best_ratio_diff:
            if area > 0.5 * image_size * image_size * ratio[0] * ratio[1]:
                best_ratio = ratio
    return best_ratio

def dynamic_preprocess(image, min_num=1, max_num=12, image_size=448, use_thumbnail=False):
    orig_width, orig_height = image.size
    aspect_ratio = orig_width / orig_height

    # calculate the existing image aspect ratio
    target_ratios = set(
        (i, j) for n in range(min_num, max_num + 1) for i in range(1, n + 1) for j in range(1, n + 1) if
        i * j <= max_num and i * j >= min_num)
    target_ratios = sorted(target_ratios, key=lambda x: x[0] * x[1])

    # find the closest aspect ratio to the target
    target_aspect_ratio = find_closest_aspect_ratio(
        aspect_ratio, target_ratios, orig_width, orig_height, image_size)

    # calculate the target width and height
    target_width = image_size * target_aspect_ratio[0]
    target_height = image_size * target_aspect_ratio[1]
    blocks = target_aspect_ratio[0] * target_aspect_ratio[1]

    # resize the image
    resized_img = image.resize((target_width, target_height))
    processed_images = []
    for i in range(blocks):
        box = (
            (i % (target_width // image_size)) * image_size,
            (i // (target_width // image_size)) * image_size,
            ((i % (target_width // image_size)) + 1) * image_size,
            ((i // (target_width // image_size)) + 1) * image_size
        )
        # split the image
        split_img = resized_img.crop(box)
        processed_images.append(split_img)
    assert len(processed_images) == blocks
    if use_thumbnail and len(processed_images) != 1:
        thumbnail_img = image.resize((image_size, image_size))
        processed_images.append(thumbnail_img)
    return processed_images

def load_image(image_file, input_size=448, max_num=12): #448
    image = Image.open(image_file).convert('RGB')
    transform = build_transform(input_size=input_size)
    images = dynamic_preprocess(image, image_size=input_size, use_thumbnail=True, max_num=max_num)
    pixel_values = [transform(image) for image in images]
    pixel_values = torch.stack(pixel_values)
    return pixel_values




In [4]:
# If you want to load a model using multiple GPUs, please refer to the `Multiple GPUs` section.
path = 'OpenGVLab/InternVL2_5-8B'
model = AutoModel.from_pretrained(
    path,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    use_flash_attn=True,
    trust_remote_code=True).eval().cuda()
tokenizer = AutoTokenizer.from_pretrained(path, trust_remote_code=True, use_fast=False)



/projects/abbott_lab/Users/ishtiaque/env/hf_models_general/lib/python3.10/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
InternLM2ForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `Pr

FlashAttention2 is not installed.


Loading checkpoint shards: 100%|██████████| 4/4 [00:05<00:00,  1.25s/it]


In [5]:
!nvidia-smi

Mon Feb 16 07:39:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          On  |   00000000:07:00.0 Off |                    0 |
| N/A   35C    P0             82W /  400W |   16511MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [6]:
# generation_config = dict(max_new_tokens=128, do_sample=True) #256 #1024

generation_config = dict(
    max_new_tokens=64,  # Only need a few tokens for single letter #10
    do_sample=False,    # Deterministic output
    temperature=0.0     # No randomness
)


In [7]:
def get_answer(answ_img_paths, query):

    query = f"<image><image><image><image>\n{query}"
    
    pixel_values_list = []
    for image_path in answ_img_paths:

        pixel_val = load_image(image_path, max_num=8).to(torch.bfloat16).cuda() #max_num=12
        pixel_values_list.append(pixel_val)
    
    pixel_values = torch.cat(pixel_values_list, dim=0)
                
    output_text = model.chat(tokenizer, pixel_values, query, generation_config, return_history=False)
    
    return output_text

In [8]:
def get_bird_images(images_folder):
# Path to the folder containing bird subfolders
    # images_folder = '/projects/abbott_lab/Users/ishtiaque/datasets/CUB_200_2011/images'
    
    # Dictionary to store bird names and their corresponding image paths
    bird_images = {}
    
    # Iterate over each subfolder in the images folder
    for folder in os.listdir(images_folder):
        bird_name = folder.split(".")[-1]  # Extract bird name from folder name
        folder_path = os.path.join(images_folder, folder)  # Path to the bird's folder
        
        # Initialize an empty list to store image paths for the current bird
        image_paths = []
        
        # Iterate over the image files in the bird's folder
        for image_file in os.listdir(folder_path):
            image_path = os.path.join(folder_path, image_file)  # Full path to the image file
            image_paths.append(image_path)  # Store the image path
        
        # Store the list of image paths in the dictionary under the bird's name
        bird_images[bird_name] = image_paths
    
    # Now bird_images contains a dictionary where the keys are bird names and the values are lists of image paths
    print(len(bird_images))
    return bird_images



In [9]:

def get_json_data(json_file_name):

    # negated_questions, modified_new_cub_class_descriptions_full_fake, #modified_new_cub_class_descriptionsx #modified_mcqs_description_only2, modified_mcqs_description_only

    # For Task 1 type 1: 
    with open(json_file_name, "r") as file: 
        json_data = json.load(file)
    print(len(json_data))
    return json_data


In [10]:
def get_map_dict(json_data):
    
    #create dictionary ishti

    captions_mcqid_pair_dict = {}
    class_freq_dict = {}
    
    for mcq in json_data:
        
        options = mcq['options'] # get the answer choices dictionary of the first mcq
        correct_option_description = {options[mcq['correct_answer']]} # get the correct answer caption
        class_name = mcq['mcq_id']
    
        # make sure that there is only one correct caption
        assert len(correct_option_description)==1, "More than one correct answer!!"
    
        #convert to string
        correct_option_description = next(iter(correct_option_description))
    
        if correct_option_description in captions_mcqid_pair_dict: # if caption already in dict
            
            # get the existing mcq_id and make sure it matches
            exist_class_name = captions_mcqid_pair_dict[correct_option_description] 
            
            # make sure mcq_id matches
            assert exist_class_name == class_name, f"mismatch in class name: [{exist_class_name}] and [{class_name}]"
            class_freq_dict [class_name] = class_freq_dict [class_name] + 1
            
    
                
        else:
            captions_mcqid_pair_dict[correct_option_description] = class_name
    
            class_freq_dict [class_name] = 1 

    return captions_mcqid_pair_dict
    
    
    # print(f"Successfully created class_names dictionary with {len(captions_mcqid_pair_dict)} Class entries from JSON file:\n {filename}.\n")
    
    # print(captions_mcqid_pair_dict)
    # print(class_freq_dict)

    

In [11]:
def get_medium_hard_data(json_data):
    
# Use this if the json file contain the medium and hard categories
    medium_data = []
    hard_data = []

    data_counter = 0
    use_partial = False#True#False
    if use_partial:
        print("using limited data for debugging")
    
    for data in json_data:
        if data['difficulty'] == "Medium":
            medium_data.append(data)
        else:
            hard_data.append(data)

        

        data_counter = data_counter + 1
        if (data_counter>50) and use_partial:
            print(f"stopping at data = {data_counter}")
            break
        
    print(len(medium_data))
    print(len(hard_data))
    return medium_data, hard_data
    

In [12]:
def run_eval(data_partition):
    

    # Counters for distribution
    true_distribution = Counter()
    predicted_distribution = Counter()
    
    results = []
    
    for i, item in tqdm(enumerate(data_partition)): # for easy part json_data, for medium_data, for hard_data 
        mcq_id = item['mcq_id']
        question = item['question']
        options = item['options']
        correct_answer = item['correct_answer']
    
        if mcq_id not in bird_images or not bird_images[mcq_id]:
            print(f"No image for {mcq_id}") 
            continue
    
        image_paths = bird_images[mcq_id][:5]
    
        # Format the prompt
        description = options[correct_answer]
        formatted_prompt = f"Which image (A, B, C or D) matches best with this description: {description}?\n"
    
    
        
    
        # Final prompt
        prompt = f""" Your answer or response must ONLY be a single index ('A', 'B', 'C', 'D'). Do not response with any other text. 
    
        {formatted_prompt}
    
        Answer: ('A', 'B', 'C', 'D')"""
    
        # Run the model
        for image_path in image_paths:
    
            #get the image paths for the four answer options
            answ_img_paths = []
            for k in ['A', 'B', 'C', 'D']:  # ['D', 'C', 'B', 'A'] for position bias checking ['A', 'B', 'C', 'D']
                # formatted_prompt += f"{k}. {options[k]}\n"
                
                if (correct_answer == k):
                    answ_img_paths.append(image_path)
                    continue
                description = options[k]
                answ_mcq_id = captions_mcqid_pair_dict[description]
                answ_mcq_img_path = bird_images[answ_mcq_id][0]
                answ_img_paths.append(answ_mcq_img_path)
                
            # print(answ_img_paths)
            
    
                
            model_output = get_answer(answ_img_paths, prompt)
            # print("Model Output: ", model_output)
    
            # Extract predicted answer (basic string search, can refine)
            predicted_answer = None
            for option in ['A', 'B', 'C', 'D']:
                if f"{option}" in model_output or f"{option}." in model_output:
                    predicted_answer = option
    
            # Update counters
            true_distribution[correct_answer] += 1
            if predicted_answer:
                predicted_distribution[predicted_answer] += 1
            
            results.append({
                'mcq_id': mcq_id,
                'image_path': image_path,
                'prompt': prompt,
                'model_output': model_output,
                'predicted_answer': predicted_answer,
                'correct_answer': correct_answer,
                'is_correct': predicted_answer == correct_answer
            })
    
    # Accuracy summary 
    print(f"Results for file: {json_file_name}")
    
    correct = sum(r['is_correct'] for r in results if r['predicted_answer'] is not None)
    total = len(results)
    print(f"Accuracy: {correct}/{total} = {correct / total:.2%}") 
    
    # Print distributions
    print("True Option Distribution:", dict(true_distribution))
    print("Predicted Option Distribution:", dict(predicted_distribution))

    return prompt, answ_img_paths

Image version--
Results for file: new_cub_with_class_descriptions.json
Accuracy: 601/1000 = 60.10%
True Option Distribution: {'C': 285, 'A': 270, 'B': 220, 'D': 225}
Predicted Option Distribution: {'C': 272, 'B': 251, 'A': 147, 'D': 330}



In [15]:
# run-3
# json_files_list = ['negated_questions', 'new_cub_class_descriptions','new_cub_with_class_descriptions', 'modified_new_cub_class_descriptions_full_fake', 'modified_new_cub_class_descriptions_partial_fake', 'incorrect_class_correct_description', 'correct_class_incorrect_description']

bird_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/CUB_200_2011/images"
food_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/food-101/food-101/images"
aircraft_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/fgvc-aircraft-2013b/data/test"
dogs_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/stanford-dogs/images/Images"
car_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/stanford-cars/train"

# bird_list = ["new_cub_class_descriptions", "new_cub_class_descriptions_task_0a_with_class_baseline"]
# food_list = ["new_food_class_descriptions", "new_food_class_descriptions_task_0a_with_class_baseline"]
# aircraft_list = ["new_aircraft_class_descriptions", "new_aircraft_class_descriptions_task_0a_with_class_baseline"]
aircraft_list = ["new_aircraft_class_descriptions_task_0a_with_class_baseline"]
dogs_list = ["new_dogs_class_descriptions", "new_dogs_class_descriptions_task_0a_with_class_baseline"]
car_list = ["new_car_class_descriptions", "new_car_class_descriptions_task_0a_with_class_baseline"]


# json_files_list = food_list  #################### change!!!!!!!!!!!!!!!!!!!!
# all_lists = [bird_list, food_list, aircraft_list, dogs_list, car_list]
# folder_lists = [bird_folder, food_folder, aircraft_folder, dogs_folder, car_folder]
all_lists = [ aircraft_list, dogs_list, car_list]
folder_lists = [aircraft_folder, dogs_folder, car_folder]

for json_files_list, images_folder in zip(all_lists, folder_lists):

    print(f"Number of JSON files: {len(json_files_list)}\n")

    bird_images = get_bird_images(images_folder)
    
    for json_file_name in json_files_list:
        
        json_file_name = f"/home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/{json_file_name}.json"
        json_data = get_json_data(json_file_name)
    
        # json_data[0]
        # json_data[1]
    
        medium_data, hard_data = get_medium_hard_data(json_data)

        captions_mcqid_pair_dict = get_map_dict(json_data)
        
        print("\n----Medium----")
        prompt, answ_img_paths = run_eval(medium_data)
        print("\n----Hard----")
        prompt, answ_img_paths = run_eval(hard_data)
    
    
    


Number of JSON files: 1

196
392
196
196

----Hard----


196it [45:25, 13.91s/it]

Results for file: /home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/new_car_class_descriptions_task_0a_with_class_baseline.json
Accuracy: 324/980 = 33.06%
True Option Distribution: {'A': 245, 'D': 250, 'B': 245, 'C': 240}
Predicted Option Distribution: {'A': 345, 'D': 41, 'C': 324, 'B': 270}


transformers_modules.OpenGVLab.InternVL2_5-8B.e9e4c0dc1db56bfab10458671519b7fa3dd29463.modeling_internvl_chat.InternVLChatModel

In [ ]:
# run-2
# json_files_list = ['negated_questions', 'new_cub_class_descriptions','new_cub_with_class_descriptions', 'modified_new_cub_class_descriptions_full_fake', 'modified_new_cub_class_descriptions_partial_fake', 'incorrect_class_correct_description', 'correct_class_incorrect_description']

bird_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/CUB_200_2011/images"
food_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/food-101/food-101/images"
aircraft_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/fgvc-aircraft-2013b/data/test"
dogs_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/stanford-dogs/images/Images"
car_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/stanford-cars/train"

# bird_list = ["new_cub_class_descriptions", "new_cub_class_descriptions_task_0a_with_class_baseline"]
# food_list = ["new_food_class_descriptions", "new_food_class_descriptions_task_0a_with_class_baseline"]
# aircraft_list = ["new_aircraft_class_descriptions", "new_aircraft_class_descriptions_task_0a_with_class_baseline"]
aircraft_list = ["new_aircraft_class_descriptions_task_0a_with_class_baseline"]
dogs_list = ["new_dogs_class_descriptions", "new_dogs_class_descriptions_task_0a_with_class_baseline"]
car_list = ["new_car_class_descriptions", "new_car_class_descriptions_task_0a_with_class_baseline"]


# json_files_list = food_list  #################### change!!!!!!!!!!!!!!!!!!!!
# all_lists = [bird_list, food_list, aircraft_list, dogs_list, car_list]
# folder_lists = [bird_folder, food_folder, aircraft_folder, dogs_folder, car_folder]
all_lists = [ aircraft_list, dogs_list, car_list]
folder_lists = [aircraft_folder, dogs_folder, car_folder]

for json_files_list, images_folder in zip(all_lists, folder_lists):

    print(f"Number of JSON files: {len(json_files_list)}\n")

    bird_images = get_bird_images(images_folder)
    
    for json_file_name in json_files_list:
        
        json_file_name = f"/home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/{json_file_name}.json"
        json_data = get_json_data(json_file_name)
    
        # json_data[0]
        # json_data[1]
    
        medium_data, hard_data = get_medium_hard_data(json_data)

        captions_mcqid_pair_dict = get_map_dict(json_data)
        
        print("\n----Medium----")
        prompt, answ_img_paths = run_eval(medium_data)
        print("\n----Hard----")
        prompt, answ_img_paths = run_eval(hard_data)
    
    
    


Number of JSON files: 1

71
140
70
70

----Medium----


0it [00:00, ?it/s]/projects/abbott_lab/Users/ishtiaque/env/hf_models_general/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
70it [17:42, 15.18s/it]


Results for file: /home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/new_aircraft_class_descriptions_task_0a_with_class_baseline.json
Accuracy: 161/350 = 46.00%
True Option Distribution: {'B': 130, 'A': 65, 'C': 80, 'D': 75}
Predicted Option Distribution: {'D': 74, 'C': 62, 'A': 170, 'B': 44}

----Hard----


70it [17:33, 15.05s/it]


Results for file: /home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/new_aircraft_class_descriptions_task_0a_with_class_baseline.json
Accuracy: 125/350 = 35.71%
True Option Distribution: {'D': 115, 'B': 75, 'A': 115, 'C': 45}
Predicted Option Distribution: {'B': 58, 'C': 138, 'D': 27, 'A': 127}
Number of JSON files: 2

120
240
120
120

----Medium----


...

196it [44:42, 13.69s/it]


Results for file: /home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/new_car_class_descriptions.json
Accuracy: 297/980 = 30.31%
True Option Distribution: {'A': 245, 'D': 250, 'B': 245, 'C': 240}
Predicted Option Distribution: {'A': 225, 'B': 317, 'C': 408, 'D': 30}
392
196
196

----Medium----


196it [43:25, 13.29s/it]


Results for file: /home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/new_car_class_descriptions_task_0a_with_class_baseline.json
Accuracy: 498/980 = 50.82%
True Option Distribution: {'B': 250, 'C': 300, 'D': 250, 'A': 180}
Predicted Option Distribution: {'A': 381, 'C': 346, 'B': 210, 'D': 43}

----Hard----


161it [37:20, 15.39s/it]

In [ ]:

# json_files_list = ['negated_questions', 'new_cub_class_descriptions','new_cub_with_class_descriptions', 'modified_new_cub_class_descriptions_full_fake', 'modified_new_cub_class_descriptions_partial_fake', 'incorrect_class_correct_description', 'correct_class_incorrect_description']

bird_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/CUB_200_2011/images"
food_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/food-101/food-101/images"
aircraft_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/fgvc-aircraft-2013b/data/test"
dogs_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/stanford-dogs/images/Images"
car_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/stanford-cars/train"

bird_list = ["new_cub_class_descriptions", "new_cub_class_descriptions_task_0a_with_class_baseline"]
food_list = ["new_food_class_descriptions", "new_food_class_descriptions_task_0a_with_class_baseline"]
aircraft_list = ["new_aircraft_class_descriptions", "new_aircraft_class_descriptions_task_0a_with_class_baseline"]
dogs_list = ["new_dogs_class_descriptions", "new_dogs_class_descriptions_task_0a_with_class_baseline"]
car_list = ["new_car_class_descriptions", "new_car_class_descriptions_task_0a_with_class_baseline"]


# json_files_list = food_list  #################### change!!!!!!!!!!!!!!!!!!!!
all_lists = [bird_list, food_list, aircraft_list, dogs_list, car_list]
folder_lists = [bird_folder, food_folder, aircraft_folder, dogs_folder, car_folder]

for json_files_list, images_folder in zip(all_lists, folder_lists):

    print(f"Number of JSON files: {len(json_files_list)}\n")

    bird_images = get_bird_images(images_folder)
    
    for json_file_name in json_files_list:
        
        json_file_name = f"/home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/{json_file_name}.json"
        json_data = get_json_data(json_file_name)
    
        # json_data[0]
        # json_data[1]
    
        medium_data, hard_data = get_medium_hard_data(json_data)

        captions_mcqid_pair_dict = get_map_dict(json_data)
        
        print("\n----Medium----")
        prompt, answ_img_paths = run_eval(medium_data)
        print("\n----Hard----")
        prompt, answ_img_paths = run_eval(hard_data)
    
    
    


Number of JSON files: 2

200
400
200
200

----Medium----


0it [00:00, ?it/s]/projects/abbott_lab/Users/ishtiaque/env/hf_models_general/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
200it [32:34,  9.77s/it]


Results for file: /home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/new_cub_class_descriptions.json
Accuracy: 745/1000 = 74.50%
True Option Distribution: {'D': 245, 'C': 220, 'B': 235, 'A': 300}
Predicted Option Distribution: {'D': 131, 'C': 202, 'A': 469, 'B': 198}

----Hard----


200it [39:40, 11.90s/it]


Results for file: /home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/new_cub_class_descriptions.json
Accuracy: 359/1000 = 35.90%
True Option Distribution: {'D': 275, 'A': 260, 'B': 215, 'C': 250}
Predicted Option Distribution: {'A': 260, 'B': 390, 'C': 321, 'D': 29}
400
200
200

----Medium----


...

70it [17:53, 15.33s/it]


Results for file: /home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/new_aircraft_class_descriptions.json
Accuracy: 99/350 = 28.29%
True Option Distribution: {'D': 115, 'B': 75, 'A': 115, 'C': 45}
Predicted Option Distribution: {'C': 212, 'D': 31, 'B': 46, 'A': 61}
140
70
70

----Medium----


70it [17:37, 15.11s/it]


Results for file: /home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/new_aircraft_class_descriptions_task_0a_with_class_baseline.json
Accuracy: 161/350 = 46.00%
True Option Distribution: {'B': 130, 'A': 65, 'C': 80, 'D': 75}
Predicted Option Distribution: {'D': 74, 'C': 62, 'A': 170, 'B': 44}

----Hard----


3it [00:44, 15.22s/it]

In [ ]:

# # json_files_list = ['negated_questions', 'new_cub_class_descriptions','new_cub_with_class_descriptions', 'modified_new_cub_class_descriptions_full_fake', 'modified_new_cub_class_descriptions_partial_fake', 'incorrect_class_correct_description', 'correct_class_incorrect_description']

# bird_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/CUB_200_2011/images"
# food_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/food-101/food-101/images"
# aircraft_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/fgvc-aircraft-2013b/data/test"
# dogs_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/stanford-dogs/images/Images"
# car_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/stanford-cars/train"

# bird_list = ["new_cub_class_descriptions",]
# food_list = ["new_food_with_class_descriptions"]
# aircraft_list = ["new_aircraft_with_class_descriptions"]
# dogs_list = ["new_dogs_with_class_descriptions"]
# car_list = ["new_car_with_class_descriptions"]


# json_files_list = food_list  #################### change!!!!!!!!!!!!!!!!!!!!
# all_lists = [food_list, aircraft_list, dogs_list, car_list]
# folder_lists = [food_folder, aircraft_folder, dogs_folder, car_folder]

# for json_files_list, images_folder in zip(all_lists, folder_lists):

#     bird_images = get_bird_images(images_folder)
    
#     for json_file_name in json_files_list:
        
#         json_file_name = f"/home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/{json_file_name}.json"
#         json_data = get_json_data(json_file_name)
    
#         # json_data[0]
#         # json_data[1]
    
#         medium_data, hard_data = get_medium_hard_data(json_data)

#         captions_mcqid_pair_dict = get_map_dict(json_data)
        
#         print("\n----Medium----")
#         prompt, answ_img_paths = run_eval(medium_data)
#         print("\n----Hard----")
#         prompt, answ_img_paths = run_eval(hard_data)
#         break
    
    
    


In [ ]:

# # json_files_list = ['negated_questions', 'new_cub_class_descriptions','new_cub_with_class_descriptions', 'modified_new_cub_class_descriptions_full_fake', 'modified_new_cub_class_descriptions_partial_fake', 'incorrect_class_correct_description', 'correct_class_incorrect_description']

# food_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/food-101/food-101/images"
# aircraft_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/fgvc-aircraft-2013b/data/test"
# dogs_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/stanford-dogs/images/Images"
# car_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/stanford-cars/train"

# food_list = ["new_food_class_descriptions", "new_food_class_descriptions_task_1a", "new_food_class_descriptions_task_1b", "new_food_class_descriptions_task_1c", "new_food_class_descriptions_task_2_full_fake", "new_food_class_descriptions_task_2_partial_fake", "task_3_negated_questions_food"]
# aircraft_list = ["new_aircraft_class_descriptions", "new_aircraft_class_descriptions_task_1a", "new_aircraft_class_descriptions_task_1b", "new_aircraft_class_descriptions_task_1c", "new_aircraft_class_descriptions_task_2_full_fake", "new_aircraft_class_descriptions_task_2_partial_fake", "task_3_negated_questions_aircraft"]
# dogs_list = ["new_dogs_class_descriptions", "new_dogs_class_descriptions_task_1a", "new_dogs_class_descriptions_task_1b", "new_dogs_class_descriptions_task_1c", "new_dogs_class_descriptions_task_2_full_fake", "new_dogs_class_descriptions_task_2_partial_fake", "task_3_negated_questions_dog"]
# car_list = ["new_car_class_descriptions", "new_car_class_descriptions_task_1a", "new_car_class_descriptions_task_1b", "new_car_class_descriptions_task_1c", "new_car_class_descriptions_task_2_full_fake", "new_car_class_descriptions_task_2_partial_fake", "task_3_negated_questions_car"]

# json_files_list = food_list  #################### change!!!!!!!!!!!!!!!!!!!!
# all_lists = [food_list, aircraft_list, dogs_list, car_list]
# folder_lists = [food_folder, aircraft_folder, dogs_folder, car_folder]

# for json_files_list, images_folder in zip(all_lists, folder_lists):

#     bird_images = get_bird_images(images_folder)
    
#     for json_file_name in json_files_list:
        
#         json_file_name = f"/home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/{json_file_name}.json"
#         json_data = get_json_data(json_file_name)
    
#         # json_data[0]
#         # json_data[1]
    
#         medium_data, hard_data = get_medium_hard_data(json_data)

#         captions_mcqid_pair_dict = get_map_dict(json_data)
        
#         print("\n----Medium----")
#         prompt, answ_img_paths = run_eval(medium_data)
#         print("\n----Hard----")
#         prompt, answ_img_paths = run_eval(hard_data)
#         break
    
    
    


In [ ]:
# # negated_questions, modified_new_cub_class_descriptions_full_fake, #modified_new_cub_class_descriptionsx #modified_mcqs_description_only2, modified_mcqs_description_only
# # new_cub_class_descriptions, new_cub_with_class_descriptions
# # json_files_list = ['new_cub_class_descriptions.json','new_cub_with_class_descriptions.json','negated_questions.json', 'modified_new_cub_class_descriptions_full_fake.json', 'modified_new_cub_class_descriptions.json', 'modified_mcqs_description_only2.json', 'modified_mcqs_description_only.json']


# json_files_list = ['new_cub_with_class_descriptions.json', 'new_cub_class_descriptions.json']

# for json_file_name in json_files_list:
#     json_data = get_json_data(json_file_name)

#     json_data[0]
#     json_data[1]

#     medium_data, hard_data = get_medium_hard_data(json_data)

#     captions_mcqid_pair_dict = get_map_dict(json_data)
    

#     prompt, answ_img_paths = run_eval(medium_data)


# print(prompt)

# print("\nImage paths are:------\n")
# for img_pths in answ_img_paths:
#     print (img_pths)